In [1]:
# Cell 1: Setup and Data Loading
import pandas as pd
import pulp 
import numpy as np
import os
import pickle


In [2]:

# Load the processed optimization inputs
data_path = '../data/processed/optimization_inputs_15_farmers.pkl'
with open(data_path, 'rb') as f:
    data = pickle.load(f)

farmers = data['farmer_ids']
practices = data['practice_ids']
c = data['constants']
BIG_M = c['BIG_M'] # This is the -1e6 value

print(f"Loaded {len(farmers)} farmers and {len(practices)} practices.")
print(f"Carbon Credit Price (CCP): ₹{c['CCP']}")

Loaded 15 farmers and 20 practices.
Carbon Credit Price (CCP): ₹1500


In [3]:
# Cell 2: Helper Cost Function
def get_shared_costs(coalition_size, total_area, constants):
    """Calculates shared MRV and Transaction costs for a coalition."""
    mrv_cost = constants['MRV_FIXED'] + constants['MRV_VARIABLE'] * (total_area ** constants['MRV_DELTA'])
    t_cost = constants['T_FIXED'] + constants['T_VARIABLE'] * coalition_size
    return mrv_cost, t_cost

In [4]:
# Cell 3: Single Farmer Optimization (Phase 1 Solo)
def optimize_solo(farmer_id, data):
    """
    Solves Phase 1 single-farmer optimization. 
    Returns the maximum net profit (Standalone Value).
    """
    prob = pulp.LpProblem(f"Solo_{farmer_id}", pulp.LpMaximize)
    
    fs = data['farm_sizes'][farmer_id]
    budget = data['farmer_budgets'][farmer_id]
    
    # 1. Decision Variables
    x = pulp.LpVariable.dicts("x", practices, cat='Binary')
    
    pairs = list(data['alpha_carbon'].keys())
    z = pulp.LpVariable.dicts("z", pairs, cat='Binary')
    
    # 2. Linearization and Incompatibility Constraints
    for (j, k) in pairs:
        # Check if this pair is marked as incompatible (-1e6) in any matrix
        is_incompatible = (
            data['alpha_carbon'].get((j,k), 0) <= -1e5 or 
            data['beta_cost'].get((j,k), 0) <= -1e5 or 
            data['gamma_yield'].get((j,k), 0) <= -1e5
        )
        
        if is_incompatible:
            # HARD CONSTRAINT: Cannot adopt both
            prob += x[j] + x[k] <= 1
            prob += z[(j, k)] == 0
        else:
            # Standard Linearization for binary multiplication (z = x_j * x_k)
            prob += z[(j, k)] <= x[j]
            prob += z[(j, k)] <= x[k]
            prob += z[(j, k)] >= x[j] + x[k] - 1
            
    # 3. Aggregates
    mrv_cost, t_cost = get_shared_costs(1, fs, c)
    
    total_csp = pulp.lpSum([x[j] * data['base_csp'][j] for j in practices]) + \
                pulp.lpSum([z[pair] * data['alpha_carbon'][pair] for pair in pairs])
                
    total_oc = pulp.lpSum([x[j] * data['base_oc'][j] for j in practices]) + \
               pulp.lpSum([z[pair] * data['beta_cost'][pair] for pair in pairs])
               
    total_y_loss = pulp.lpSum([z[pair] * data['gamma_yield'].get(pair, 0) for pair in pairs])
    
    # 4. Budget Constraint
    # Total operational cost + yield loss + certification costs must be within budget
    prob += total_oc + (fs * total_y_loss) + mrv_cost + t_cost <= budget
    
    # 5. Objective Function: Maximize Net Profit
    carbon_revenue = fs * total_csp * c['CCP']
    net_profit = carbon_revenue - (fs * total_y_loss) - total_oc - mrv_cost - t_cost
    prob += net_profit
    
    # 6. Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # If the farmer can't afford the MRV cost or makes a loss, their standalone value is 0 (they opt out)
    if pulp.LpStatus[prob.status] == 'Optimal' and pulp.value(prob.objective) > 0:
        return pulp.value(prob.objective)
    return 0.0

In [5]:
# Cell 4: Execute Solo Optimization
print("--- Standalone Values (Individual Rationality) ---")
standalone_values = {}

for f_id in farmers:
    val = optimize_solo(f_id, data)
    standalone_values[f_id] = val
    print(f"Farmer {f_id} (Area: {data['farm_sizes'][f_id]} ha, Budget: ₹{data['farmer_budgets'][f_id]:.0f}) -> Solo Value: ₹ {val:,.2f}")

# Save for Phase 2
with open('../data/processed/standalone_values_15_farmers.pkl', 'wb') as f:
    pickle.dump(standalone_values, f)
    
total_solo_value = sum(standalone_values.values())
print(f"\nTotal System Value if all act independently: ₹ {total_solo_value:,.2f}")

--- Standalone Values (Individual Rationality) ---
Farmer F0001 (Area: 0.82 ha, Budget: ₹3032) -> Solo Value: ₹ 0.00
Farmer F0002 (Area: 0.21 ha, Budget: ₹698) -> Solo Value: ₹ 0.00
Farmer F0003 (Area: 1.28 ha, Budget: ₹4767) -> Solo Value: ₹ 0.00
Farmer F0004 (Area: 1.55 ha, Budget: ₹5075) -> Solo Value: ₹ 0.00
Farmer F0005 (Area: 0.69 ha, Budget: ₹2347) -> Solo Value: ₹ 0.00
Farmer F0006 (Area: 0.44 ha, Budget: ₹1510) -> Solo Value: ₹ 0.00
Farmer F0007 (Area: 0.6 ha, Budget: ₹2016) -> Solo Value: ₹ 0.00
Farmer F0008 (Area: 0.26 ha, Budget: ₹820) -> Solo Value: ₹ 0.00
Farmer F0009 (Area: 1.46 ha, Budget: ₹4840) -> Solo Value: ₹ 0.00
Farmer F0010 (Area: 1.32 ha, Budget: ₹4470) -> Solo Value: ₹ 0.00
Farmer F0011 (Area: 0.65 ha, Budget: ₹2190) -> Solo Value: ₹ 0.00
Farmer F0012 (Area: 1.87 ha, Budget: ₹6919) -> Solo Value: ₹ 0.00
Farmer F0013 (Area: 0.97 ha, Budget: ₹3088) -> Solo Value: ₹ 0.00
Farmer F0014 (Area: 0.26 ha, Budget: ₹890) -> Solo Value: ₹ 0.00
Farmer F0015 (Area: 0.88 ha, 